In [ ]:
from sys import hash_info

from openai import OpenAI
import gradio as gr
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

In [ ]:
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
openai = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [ ]:
MODEL = "gemini-3.1-flash-lite"
system_message = "You are a helpful assistant"

In [ ]:
def chat(message, history):
  history = [{"role":h["role"], "content":h["content"]} for h in history]
  messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
  response = openai.chat.completions.create(model=MODEL, messages=messages)
  return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat).launch()

In [ ]:
def chat(message, history):
  history = [{"role":h["role"], "content":h["content"]} for h in history]
  messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
  stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
  response = ""
  for chunk in stream:
      response+=chunk.choices[0].delta.content or ''
      yield response

In [ ]:
gr.ChatInterface(fn=chat).launch()

In [ ]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [ ]:
gr.ChatInterface(fn=chat).launch()

In [ ]:
system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [ ]:
gr.ChatInterface(fn=chat).launch()

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'belt' in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."

    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
gr.ChatInterface(fn=chat).launch()